# Topic 7 — Model-Free RL: GridWorld and the Experience Stream

In a **known MDP** we could read the reward $R(s,a)$ and transition $P(s'\mid s,a)$ from a table and *compute* values.
In **model-free** RL those are hidden: the agent learns only from a stream of experiences `(s, a, r, s')`.

This notebook builds the `GridWorld` environment and a random agent, then shows what that experience stream looks like.
The two update rules that turn this stream into a value table come next: **Monte Carlo** (Topic 8) and **Temporal Difference** (Topic 9).


## The environment
The model (rewards, movement rules) lives *inside* `GridWorld`. The agent only ever sees what `step()` returns: `(next_state, reward, done)`.


In [ ]:
class GridWorld():
    def __init__(self):
        self.x = 0   # row (0 = top)
        self.y = 0   # col (0 = left)

    def step(self, a):
        if a == 0:   self.move_left()
        elif a == 1: self.move_up()
        elif a == 2: self.move_right()
        elif a == 3: self.move_down()
        reward = -1                 # each step costs 1
        done = self.is_done()
        return (self.x, self.y), reward, done

    def move_right(self): self.y = min(self.y + 1, 3)
    def move_left(self):  self.y = max(self.y - 1, 0)
    def move_up(self):    self.x = max(self.x - 1, 0)
    def move_down(self):  self.x = min(self.x + 1, 3)

    def is_done(self):
        return self.x == 3 and self.y == 3   # goal at bottom-right

    def reset(self):
        self.x, self.y = 0, 0
        return (self.x, self.y)


## A random agent
With no knowledge yet, wandering uniformly at random is the honest start.


In [ ]:
import random

class Agent():
    def select_action(self):
        return random.randint(0, 3)   # 0=left, 1=up, 2=right, 3=down


## What one experience looks like
Put the agent at the start and take a few chosen actions. Each `step` returns **one experience sample** — all a model-free method ever gets.


In [ ]:
ACTIONS = {0: "left", 1: "up", 2: "right", 3: "down"}

env = GridWorld()
s = env.reset()
print(f"start state: {s}\n")
for a in [2, 3, 3, 2]:                 # right, down, down, right
    s_prev = (env.x, env.y)
    s_next, r, done = env.step(a)
    print(f"step({s_prev}, {ACTIONS[a]:>5}) -> next_state {s_next}, reward {r}, done {done}")


## Run full random episodes
Model-free methods need *many* episodes. Here we just measure how long a random walk takes to reach the goal (it is slow — that is why the true value of the start state is about $-59$).


In [ ]:
env = GridWorld()
agent = Agent()

for ep in range(3):
    env.reset()
    steps, total_reward, done = 0, 0, False
    while not done:
        a = agent.select_action()
        (x, y), r, done = env.step(a)
        steps += 1
        total_reward += r
    print(f"episode {ep+1}: reached goal in {steps} steps, return G = {total_reward}")


## Next
This is the *environment* half of model-free RL. The *learning* half — turning experience into a value table — is **Monte Carlo** (Topic 8, average the real returns) and **Temporal Difference** (Topic 9, update every step toward reward + estimate of the next state).
